In [4]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

# --- UTILITY: QUANTUM RANDOM GENERATOR ---
def get_quantum_random_bits(n):
    bits = []
    backend = BasicSimulator()
    for _ in range(n):
        qc = QuantumCircuit(1)
        qc.h(0)
        qc.measure_all()
        # Transpile and run
        t_qc = transpile(qc, backend)
        result = backend.run(t_qc, shots=1).result()
        counts = result.get_counts()
        bits.append(int(list(counts.keys())[0]))
    return bits

# --- PARAMETERS ---
n_qubits = 20 
threshold = 0.1 

# --- ALICE: PREPARATION  ---
alice_bits = get_quantum_random_bits(n_qubits)
alice_bases = get_quantum_random_bits(n_qubits) # 0 = Z-basis, 1 = X-basis

qubit_list = []
for i in range(n_qubits):
    qc = QuantumCircuit(1)
    if alice_bits[i] == 1:
        qc.x(0)
    if alice_bases[i] == 1: 
        qc.h(0)
    qubit_list.append(qc)

# --- BOB: MEASUREMENT  ---
bob_bases = get_quantum_random_bits(n_qubits)
bob_results = []
backend = BasicSimulator()

for i in range(n_qubits):
    qc = qubit_list[i]
    if bob_bases[i] == 1:
        qc.h(0)
    qc.measure_all()
    
    t_qc = transpile(qc, backend)
    result = backend.run(t_qc, shots=1).result()
    counts = result.get_counts()
    bob_results.append(int(list(counts.keys())[0]))

# --- SIFTING (PUBLIC CHANNEL) ---
alice_key = []
bob_key = []

for i in range(n_qubits):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_results[i])

# --- ERROR CHECKING & REPORTING ---
if len(alice_key) > 0:
    errors = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    error_rate = errors / len(alice_key)
    
    print(f"Sifted Key length: {len(alice_key)}")
    print(f"Error Rate: {error_rate * 100}%")
    
    if error_rate > threshold:
        print("ALERT: Possible Attacker Detected!") 
    else:
        print("Success: Secure key established.")
else:
    print("No matching bases found.")

Sifted Key length: 10
Error Rate: 0.0%
Success: Secure key established.
